In [0]:
import requests

app_id = dbutils.secrets.get(scope="adzuna", key="app_id")
app_key = dbutils.secrets.get(scope="adzuna", key="app_key")

response = requests.get(
    "https://api.adzuna.com/v1/api/jobs/gb/search/1",
    params={"app_id": app_id, "app_key": app_key, "results_per_page": 1, "what": "data analyst"},
    timeout=30,
)

print(response.status_code)
print(response.json())

In [0]:
def fetch_adzuna_page(search_term, page, country="gb", results_per_page=50, max_days_old=None, title_only=False):
    params = {
        "app_id": app_id,
        "app_key": app_key,
        "results_per_page": results_per_page,
        "what": search_term,
        "content-type": "application/json",
    }
    if max_days_old is not None:
        params["max_days_old"] = max_days_old
    if title_only:
        params["title_only"] = 1

    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

test = fetch_adzuna_page("data", page=1, title_only=True)
print(f"Total jobs with 'data' in title: {test.get('count')}")
print(f"Jobs on this page: {len(test.get('results', []))}")

In [0]:
def fetch_all_pages(search_term, country="gb", max_days_old=None, 
                    title_only=False, max_pages=100):
    page = 1
    all_results = []

    while page <= max_pages:
        print(f"  Fetching page {page}...")
        data = fetch_adzuna_page(
            search_term, page,
            country=country,
            max_days_old=max_days_old,
            title_only=title_only
        )
        results = data.get("results", [])

        if not results:
            print(f"  No results on page {page} — done.")
            break

        all_results.extend(results)
        print(f"  Got {len(results)} jobs (running total: {len(all_results)})")
        page += 1
        time.sleep(1)

    return all_results

results = fetch_all_pages("data", title_only=True)
print(f"\nFinished. Total jobs pulled: {len(results)}")

Backfilling all data jobs starting today 22-06-2026

In [0]:
import json
from datetime import date

def write_to_volume(results, search_term, catalog, run_type="daily"):
    today = date.today().isoformat()

    payload = {
        "pulled_at": today,
        "search_term": search_term,
        "country": "gb",
        "run_type": run_type,
        "total_results": len(results),
        "results": results
    }

    file_name = f"adzuna_{run_type}_{search_term.replace(' ', '_')}_{today}.json"
    path = f"/Volumes/workspace/job_postings/raw_landings/{file_name}"

    dbutils.fs.put(path, json.dumps(payload, indent=2), overwrite=True)
    print(f"Written {len(results)} jobs to {path}")

write_to_volume(results, "data", "workspace","backfill")